# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset, "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells", using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL. All entities (record sets, fields, etc.) are referenced by their `@id`.

In [ ]:
# Install required library
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and instantiate the dataset object using the Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant Schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
md = dataset.metadata
print(f"{md.name}: {md.description}")
print(f"Identifier: {md.identifier}\nVersion: {md.version}\nLicense: {md.license}")

## 2. Data Overview
Review available record sets, their `@id`s, and associated fields. This is important for referencing and extracting data correctly from the package.

In [ ]:
# List all record sets and their fields by @id
from pprint import pprint

print("Available record sets and their fields:")
for rs in dataset.record_sets:
    print(f"- RecordSet name: {rs.name} | @id: {rs.id}")
    for field in rs.fields:
        print(f"    - Field name: {field.name} | @id: {field.id} | dataType: {field.data_type}")
    print("----")

## 3. Data Extraction
Extract all records from each record set and load them as Pandas DataFrames. Use the `@id` of the record set for all references.

In [ ]:
# Gather all record set @id values
record_set_ids = [rs.id for rs in dataset.record_sets]

# Create a DataFrame for each record set
dataframes = {}
for rset_id in record_set_ids:
    records = list(dataset.records(record_set=rset_id))
    df = pd.DataFrame(records)
    dataframes[rset_id] = df

# Show the first record set's columns and first few records for reference
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"Columns for {first_rs}:\n", dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate EDA using a quantitative field from a record set, following typical recipes: filtering, normalization, and grouping. Reference all columns by their field `@id` for consistent usage. **Update numeric_field_id and group_field_id below to match actual field `@id`s from your data overview above.**

In [ ]:
# Select a record set and field @id for quantitative analysis
# Replace these IDs below with those listed in the Data Overview section
record_set_id = record_set_ids[0] if record_set_ids else None
# Inspect columns to select a numeric field for analysis
if record_set_id:
    cols = dataframes[record_set_id].columns.tolist()
    print('Columns:', cols)
    # Example: Try to pick an obvious quantitative column; adjust as needed
    # Suppose 'cr:MW' for Molecular Weight, and 'cr:Gene_Name' for grouping
    example_numeric = next((c for c in cols if 'MW' in c or 'weight' in c.lower() or 'cr:MW' == c), cols[0])
    example_group = next((c for c in cols if 'gene' in c.lower() or 'sample' in c.lower()), None)
    print(f"Selected numeric field: {example_numeric}")
    if example_group:
        print(f"Selected group field: {example_group}")

    # Remove non-numeric values that may have slipped in
    df = dataframes[record_set_id].copy()
    df[example_numeric] = pd.to_numeric(df[example_numeric], errors='coerce')

    # Filter based on threshold (demonstration: MW > 10000)
    threshold = 10000
    filtered_df = df[df[example_numeric] > threshold]
    print(f"Filtered records with {example_numeric} > {threshold}:")
    display(filtered_df.head())

    # Normalize the field
    filtered_df[f"{example_numeric}_normalized"] = (
        (filtered_df[example_numeric] - filtered_df[example_numeric].mean()) /
        filtered_df[example_numeric].std()
    )
    print(f"Normalized {example_numeric} for filtered records:")
    display(filtered_df[[example_numeric, f"{example_numeric}_normalized"]].head())

    # Group by another field if it exists
    if example_group and example_group in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(example_group)[example_numeric]
            .mean()
            .reset_index()
            .sort_values(by=example_numeric, ascending=False)
        )
        print(f"Grouped data by {example_group} (mean {example_numeric}):")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between selected numeric and categorical fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple histogram and boxplot for the selected numeric field
if record_set_id and example_numeric:
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    sns.histplot(df[example_numeric].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {example_numeric}')
    
    plt.subplot(1,2,2)
    sns.boxplot(y=df[example_numeric])
    plt.title(f'Boxplot of {example_numeric}')
    plt.show()

# If grouping field is available, show top mean by group
if record_set_id and example_group:
    top_groups = (
        df.groupby(example_group)[example_numeric].mean().sort_values(ascending=False).head(10)
    )
    plt.figure(figsize=(8,6))
    sns.barplot(x=top_groups.values, y=top_groups.index, orient='h')
    plt.xlabel(f'Mean {example_numeric}')
    plt.ylabel(example_group)
    plt.title(f'Top 10 groups by mean {example_numeric}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use `mlcroissant` to load, review, and analyze a FAIR²-compliant experimental dataset. All dataset entities—record sets, fields, and columns—were referenced via their `@id` ensuring unambiguous, standards-based data processing. Exploratory steps included filtering, normalization, and visualization applied to a molecular weight or abundance field, with grouping by gene or sample if available. You can extend this analysis to other record sets, fields, and processing protocols according to your research needs.